# Calculate Vibrational Energies for the Ground States

In [20]:
import psi4
import csv
import numpy as np
from psi4.driver.diatomic import anharmonicity

molecule = "h2plus"            #molecule, h2 or h2+
method = "fci"                  #method, hf, ccsd, fci
npoints = 10                    #number of points to use around the minimum
nenergies = 20                 #number of vibrational energies to calculate

file = f"{molecule}_{method}_pec-Copy2.csv"           #file name

csv_out = f"{molecule}_{method}_vib.csv"
psi4_out = f"psi4_{molecule}_{method}_vib.out"
psi4.set_output_file(psi4_out, False)

#function to read file
def read_two_col_csv(path):                                      
    R, E = [], []                               #create empty lists to hold distances and energies
    with open(path, "r", newline="") as f:      #open file
        reader = csv.reader(f)                  #create object to read file
        _ = next(reader)  # header              #skips header row
        for row in reader:
            if not row:
                continue
            R.append(float(row[0]))             #add R to list
            E.append(float(row[1]))             #add each E to list
    return np.array(R), np.array(E)             #return lists as arrays

R_values, energies = read_two_col_csv(file)     #run function for chosen file

# Select points around the minimum
imin = np.argmin(energies)                      #finds index for the minimum energy
#energies = energies - np.min(energies)          #shift all energies by the minimum
left = max(0, imin - npoints)                   #sets left endpoint for fit min(0,5) ensures you don't try to take points before 0
right = min(len(R_values), imin + npoints + 1)  # +1 because Python slices exclude the endpoint

R_fit = R_values[left:right]                    #makes list of fit distances 
E_fit = energies[left:right]                    #makes list of fit energies

#psi4 only needs it to get the molecule properties, ignores the distance, so 0.74 is arbitrary
h2plus = psi4.geometry("""
1 2
H
H 1 0.74           
symmetry c1
units angstrom
""")

h2 = psi4.geometry("""
0 1
H
H 1 0.74           
symmetry c1
units angstrom
""")

#create a dictionary with possible molecule names
molecules = {"h2plus":h2plus, "h2": h2}

#run anharmonicity calculation
spec = anharmonicity(R_fit.tolist(), E_fit.tolist(), mol=molecules[molecule])

we = spec["we"]
wexe = spec["wexe"]

print(f"PEC minimum = {spec['re']} Angstroms")
print(f"Vibrationally averaged minimum = {spec['r0']} Angstroms")
print(f"Energy at minimun = {spec['E(re)']} hartree")
print(f"Harmonic vibrational frequency = {spec['we']} cm^-1")
print(f"Anharmonicity constant = {spec['wexe']} cm^-1")
print(f"Fundamental transition = {spec['nu']} cm^-1")
print(f"Zero Point Energy (harmonic) = {spec['ZPVE(harmonic)']} cm^-1")
print(f"Zero Point Energy (anharmonic) = {spec['ZPVE(anharmonic)']} cm^-1")
print(f"Rotational at equilibrium = {spec['Be']} cm^-1")
print(f"Rotational in vibrational ground state = {spec['B0']} cm^-1")
print(f"Rotation-vibration coupling = {spec['ae']} cm^-1")
print(f"Centrifugal distortion constant = {spec['De']} cm^-1")

#calculate vibrational energies
#we(n + 1/2) is the harmonic oscillator approximation
#-wexe(n + 1/2)^2 is the anharmonic correction, comes from solving Schrodinger's equation for the morse potential
Gv_values = []
for v in range(nenergies +1):
    Gv = we * (v + 0.5) - wexe * (v + 0.5)**2 
    Gv_values.append(Gv)
    print(f"v={v}: G({v}) = {Gv:.4f} cm^-1")

spec_keys = list(spec.keys())
spec_values = list(spec.values())

max_len = max(len(Gv_values), len(spec_keys))

Gv_values += [np.nan] * (max_len - len(Gv_values))
spec_keys += [np.nan] * (max_len - len(spec_keys))
spec_values += [np.nan] * (max_len - len(spec_values))


#write csv file
with open(csv_out, "w", newline="") as f:                 #opens csv file, overwrite existing file, prevent blank lines
    w = csv.writer(f)                                     #make csv writer
    w.writerow(["Gv (cm^-1)", "Variable", "Value"])  #write a header row
    
    for i in range(max_len):
        w.writerow([Gv_values[i], spec_keys[i], spec_values[i]])

print(f"\nSaved {csv_out}  |  Psi4 output: {psi4_out}")   #print to show completion

    

PEC minimum = 1.056884425932756 Angstroms
Vibrationally averaged minimum = 1.0713529425676007 Angstroms
Energy at minimun = -0.602622666911583 hartree
Harmonic vibrational frequency = 2323.6177574059225 cm^-1
Anharmonicity constant = 67.41679251620981 cm^-1
Fundamental transition = 2188.7841723735028 cm^-1
Zero Point Energy (harmonic) = 1161.8088787029612 cm^-1
Zero Point Energy (anharmonic) = 1144.9546805739087 cm^-1
Rotational at equilibrium = 29.9492778150461 cm^-1
Rotational in vibrational ground state = 29.145815904213865 cm^-1
Rotation-vibration coupling = 1.6069238216644681 cm^-1
Centrifugal distortion constant = 0.019901677353202925 cm^-1
v=0: G(0) = 1144.9547 cm^-1
v=1: G(1) = 3333.7389 cm^-1
v=2: G(2) = 5387.6894 cm^-1
v=3: G(3) = 7306.8064 cm^-1
v=4: G(4) = 9091.0899 cm^-1
v=5: G(5) = 10740.5397 cm^-1
v=6: G(6) = 12255.1559 cm^-1
v=7: G(7) = 13634.9386 cm^-1
v=8: G(8) = 14879.8877 cm^-1
v=9: G(9) = 15990.0032 cm^-1
v=10: G(10) = 16965.2851 cm^-1
v=11: G(11) = 17805.7334 cm^-